# Import Libraries

In [18]:
import pandas as pd
import numpy as np
import spacy
import matplotlib.pyplot as plt
import os
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import torch
import re
import tensorflow as tf
from models.llama3.generation import Llama
from tqdm import tqdm 
from sklearn.metrics import classification_report

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

In [19]:
label_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

In [20]:
q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/balanced_dataset.csv')
queries = q_df['question']
label = q_df['label'].str.lower().map(label_mapper)

# Question Classifier

## Text-Classification

In [2]:
tokenizer = AutoTokenizer.from_pretrained("uw-vta/bloominzer-0.1")
model = AutoModelForSequenceClassification.from_pretrained("uw-vta/bloominzer-0.1")
bloominzer = pipeline("text-classification", model=model, tokenizer=tokenizer)

Device set to use mps:0


In [3]:
print(bloominzer("If I have 2 pair of apple, can i make apple pie with it?"))

[{'label': 'Synthesis', 'score': 0.9990537762641907}]


## LLM Classification

## Zero-Shot

### BART

In [4]:
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-mnli")
model = AutoModelForSequenceClassification.from_pretrained("facebook/bart-large-mnli")

bart = pipeline("zero-shot-classification",
                      model=model , tokenizer=tokenizer)

Device set to use mps:0


In [5]:
sequence_to_classify = "If I have 2 pair of apple, can i make apple pie with it?"
candidate_labels = ['knowledge', 'comprehension', 'application', 'analysis','synthesis', 'evaluation']
label = bart(sequence_to_classify, candidate_labels)

In [6]:
label['labels'][0]

'application'

### mDeBERTa-v3-base-mnli-xnli

In [7]:
tokenizer = AutoTokenizer.from_pretrained("MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")
model = AutoModelForSequenceClassification.from_pretrained("MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

ya_classifier = pipeline("zero-shot-classification",
                      model=model , tokenizer=tokenizer)

Device set to use mps:0


In [ ]:
sequence_to_classify = "If I have 2 pair of apple, can i make apple pie with it?"
candidate_labels = ['knowledge', 'comprehension', 'application', 'analysis','synthesis', 'evaluation']
label = ya_classifier(sequence_to_classify, candidate_labels)

In [11]:
label['labels'][0]

'application'

## Text Generation

### LLAMA

In [1]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
	repo_id="modularai/Llama-3.1-8B-Instruct-GGUF",
	filename="llama-3.1-8b-instruct-q4_k_m.gguf"
)


/opt/anaconda3/envs/yt_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
llama_model_load_from_file_impl: using device Metal (Apple M3 Pro) - 12287 MiB free
llama_model_loader: loaded meta data with 33 key-value pairs and 292 tensors from /Users/pranitdas/.cache/huggingface/hub/models--modularai--Llama-3.1-8B-Instruct-GGUF/snapshots/966694508430d1177f6d585de779250e7a34bc3a/./llama-3.1-8b-instruct-q4_k_m.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str   

In [16]:
query = 'If I have 2 pair of apple, can i make apple pie with it?'

message = llm.create_chat_completion(
    messages=[
        {
            "role": "user",
            "content": f"""
        Classify the following educational query into the appropriate Bloom's taxonomy level using these definitions:

        **Bloom's Taxonomy Levels:**
        1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
        - Keywords: define, list, memorize, recall, repeat
        2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
        - Keywords: describe, explain, paraphrase, summarize
        3. Application: Using learned information in new concrete situations to solve problems
        - Keywords: apply, demonstrate, solve, use, implement
        4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
        - Keywords: analyze, compare, contrast, differentiate, examine
        5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
        - Keywords: create, design, propose, formulate, integrate
        6. Evaluation: Making judgments based on criteria and standards through checking and critiquing
        - Keywords: assess, critique, defend, evaluate, justify

        **Classification Rules:**
        - Output ONLY the lowercase label name (knowledge, comprehension, application, analysis, synthesis, evaluation)
        - Choose the HIGHEST level that substantially applies
        - Ignore verb tense/stemming (e.g., "analyzed" = analysis)
        - If multiple levels apply, select the most complex

        **Query to Classify:**
        "{query}"
    """
        }
    ]
)

print(message['choices'][0]['message']['content'])

Llama.generate: 370 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    7564.58 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =     124.69 ms /     2 runs   (   62.35 ms per token,    16.04 tokens per second)
llama_perf_context_print:       total time =     126.28 ms /     3 tokens


analysis


In [21]:
pred_labels= []
for query in tqdm(queries):
    message = llm.create_chat_completion(
        messages=[
            {
                "role": "user",
                "content": f"""
            Classify the following educational query into the appropriate Bloom's taxonomy level using these definitions:

            **Bloom's Taxonomy Levels:**
            1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
            - Keywords: define, list, memorize, recall, repeat
            2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
            - Keywords: describe, explain, paraphrase, summarize
            3. Application: Using learned information in new concrete situations to solve problems
            - Keywords: apply, demonstrate, solve, use, implement
            4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
            - Keywords: analyze, compare, contrast, differentiate, examine
            5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
            - Keywords: create, design, propose, formulate, integrate
            6. Evaluation: Making judgments based on criteria and standards through checking and critiquing
            - Keywords: assess, critique, defend, evaluate, justify

            **Classification Rules:**
            - Output ONLY the lowercase label name (knowledge, comprehension, application, analysis, synthesis, evaluation)
            - Choose the HIGHEST level that substantially applies
            - Ignore verb tense/stemming (e.g., "analyzed" = analysis)
            - If multiple levels apply, select the most complex

            **Query to Classify:**
            "{query}"
        """
            }
        ]
    )
    pred_labels.append(message['choices'][0]['message']['content'].lower())

  0%|          | 0/600 [00:00<?, ?it/s]Llama.generate: 47 prefix-match hit, remaining 325 prompt tokens to eval
llama_perf_context_print:        load time =    7564.58 ms
llama_perf_context_print: prompt eval time =    5324.22 ms /   325 tokens (   16.38 ms per token,    61.04 tokens per second)
llama_perf_context_print:        eval time =      44.88 ms /     1 runs   (   44.88 ms per token,    22.28 tokens per second)
llama_perf_context_print:       total time =    5370.71 ms /   326 tokens
  0%|          | 1/600 [00:05<53:39,  5.37s/it]Llama.generate: 349 prefix-match hit, remaining 17 prompt tokens to eval
llama_perf_context_print:        load time =    7564.58 ms
llama_perf_context_print: prompt eval time =     444.21 ms /    17 tokens (   26.13 ms per token,    38.27 tokens per second)
llama_perf_context_print:        eval time =    1401.60 ms /    31 runs   (   45.21 ms per token,    22.12 tokens per second)
llama_perf_context_print:       total time =    1851.99 ms /    48 token

In [22]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

KeyError: 'comprehension\n\nthis query requires the student to demonstrate understanding of the cubist movement by listing its unique characteristics, which is a key aspect of comprehension.'

### OWEN

In [7]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

qwen_classifier = pipeline("text-generation", model=model , tokenizer=tokenizer)

Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.32s/it]
Device set to use mps:0


In [ ]:
query = 'What is the capital of india?'

messages = [
    {
        "role": "user", 
        "content": f'''
        Classify this query's Bloom's taxanomy level using ONLY one word from: 
        [knowledge, comprehension, application, analysis, synthesis, evaluation]
        Respond ONLY with the exact taxonomy level name. No explanations
        query: {query}.''',
    }
]
message = qwen_classifier(messages)

print(message[0]['generated_text'][1]['content'])

### Predict Label Assignment

In [ ]:
# Reinforce 1
def q_classifier(query, prev_res):
    messages = [
        {
            "role": "user", 
            "content": f"""
            REVISE YOUR CLASSIFICATION. Your previous response '{prev_res}' was INVALID. 
            Classify this query's Bloom's taxonomy level using ONLY one word from: 
            [knowledge, comprehension, application, analysis, synthesis, evaluation]

            CRITICAL RULES:
            1. MUST select from the 6 specified terms - NO exceptions
            2. Use these precise definitions:
            - knowledge: Recalling facts, terms, basic concepts (identify, list, name)
            - comprehension: Explaining meaning (describe, discuss, summarize)
            - application: Using information in new situations (execute, implement, solve)
            - analysis: Drawing connections among ideas (differentiate, organize, attribute)
            - synthesis: Producing new patterns (design, construct, integrate)
            - evaluation: Making judgments with evidence (appraise, defend, recommend)
            3. If multiple levels apply, choose the HIGHEST appropriate level
            4. Respond ONLY with the lowercase taxonomy word - NO other text

            Query: "{query}"

            Re-evaluate carefully. Your response MUST be exactly one word from the list.
            """
        }
    ]

    message = qwen_classifier(messages)
    
    return message[0]['generated_text'][1]['content']

In [ ]:
# Reinforce 2
def q1_classifier(query, prev_res):
    messages = [
        {
            "role": "user", 
            "content": f"""
            Predict Bloom's Taxonomy level of understanding. Classify query: {query} in one word. Responding {prev_res} is danger.
            Your entire response must be just one word chosen from following: 
            {label_mapper.keys()}
            {prev_res} is incorrect.
            """
        }
    ]

    message = qwen_classifier(messages)
    
    return message[0]['generated_text'][1]['content']

In [119]:
pred_labels= []
for query in tqdm(queries):
    messages = [
        {
            "role": "user", 
            "content": f'''
            Classify the query's Bloom's taxonomy level using ONLY one word from: 
            [knowledge, comprehension, application, analysis, synthesis, evaluation]

            Consider these definitions:
            1. **knowledge**: Recalling facts/definitions (who, what, when, where)
            2. **comprehension**: Explaining concepts in own words (summarize, describe)
            3. **application**: Using knowledge in new situations (solve, compute, demonstrate)
            4. **analysis**: Breaking down concepts (compare, contrast, categorize)
            5. **synthesis**: Creating new patterns/solutions (design, develop, integrate)
            6. **evaluation**: Making judgments with criteria (justify, critique, recommend)

            Query: "{query}"

            Decision rules:
            - Focus on the query's PRIMARY cognitive demand
            - For multiple operations, choose the HIGHEST applicable level

            Respond ONLY with the exact taxonomy word in lowercase. No punctuation.
            ''',
        }
    ]

    message = qwen_classifier(messages)
    
    pred_labels.append(message[0]['generated_text'][1]['content'])

while(len(set(pred_labels)) != 6):
    for i , query in enumerate(queries):
        if(pred_labels[i].lower() not in label_mapper.keys()):
            prev_res = pred_labels[i]
            print(prev_res)
            pred_labels[i] = q_classifier(query, prev_res)
            if(pred_labels[i].lower() not in label_mapper.keys()):
                pred_labels[i] = q1_classifier(query, prev_res)
    print(set(pred_labels))

100%|██████████| 600/600 [08:16<00:00,  1.21it/s]


composition
explanation
comparison
definition
definition
definition
comparison
comparison
recognition
definition
definition
definition
{'comprehension', 'synthesis', 'comparison is evaluation.', 'knowledge', 'analysis', 'application', 'evaluation', 'definition is incorrect.'}
comparison is evaluation.
definition is incorrect.
comparison is evaluation.
{'comprehension', 'synthesis', 'knowledge', 'analysis', 'application', 'evaluation', 'definition'}
definition
{'comprehension', 'synthesis', 'knowledge', 'analysis', 'application', 'evaluation', 'definition is incorrect.'}
definition is incorrect.
{'comprehension', 'synthesis', 'knowledge', 'analysis', 'application', 'evaluation', 'definition'}
definition
{'comprehension', 'synthesis', 'knowledge', 'analysis', 'application', 'evaluation', 'definition'}
definition
{'comprehension', 'synthesis', 'knowledge', 'analysis', 'application', 'evaluation', 'definition is incorrect.'}
definition is incorrect.
{'comprehension', 'synthesis', 'knowledg

In [120]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

              precision    recall  f1-score   support

           0       1.00      0.51      0.68       100
           1       0.72      0.43      0.54       100
           2       0.34      0.69      0.46       100
           3       0.60      0.79      0.68       100
           4       0.77      0.56      0.65       100
           5       0.86      0.72      0.78       100

    accuracy                           0.62       600
   macro avg       0.71      0.62      0.63       600
weighted avg       0.71      0.62      0.63       600



## Google-FLAN-T5-XL

In [4]:
model_name = "google/flan-t5-xl"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

flan_classifier = pipeline("text2text-generation", model=model , tokenizer=tokenizer, device=-1)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  9.74it/s]
Device set to use cpu


In [ ]:
query = 'What is the capital of india?'

messages = f"""
    Classify the following educational query into the appropriate Bloom's taxonomy level using these definitions:

    **Bloom's Taxonomy Levels:**
    1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
    - Keywords: define, list, memorize, recall, repeat
    2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
    - Keywords: describe, explain, paraphrase, summarize
    3. Application: Using learned information in new concrete situations to solve problems
    - Keywords: apply, demonstrate, solve, use, implement
    4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
    - Keywords: analyze, compare, contrast, differentiate, examine
    5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
    - Keywords: create, design, propose, formulate, integrate
    6. Evaluation: Making judgments based on criteria and standards through checking and critiquing
    - Keywords: assess, critique, defend, evaluate, justify

    **Classification Rules:**
    - Output ONLY the lowercase label name (knowledge, comprehension, application, analysis, synthesis, evaluation)
    - Choose the HIGHEST level that substantially applies
    - Ignore verb tense/stemming (e.g., "analyzed" = analysis)
    - If multiple levels apply, select the most complex

    **Query to Classify:**
    "{query}"
        """

message = flan_classifier(messages)
print(message)

### Predict Label Assignment

In [15]:
pred_labels= []
for query in tqdm(queries):
    messages = f"""
        Classify the following educational query into the appropriate Bloom's taxonomy level using these definitions:

        **Bloom's Taxonomy Levels:**
        1. Knowledge: Recalling facts, terms, basic concepts, or answers without necessarily understanding them
        - Keywords: define, list, memorize, recall, repeat
        2. Comprehension: Demonstrating understanding of facts by interpreting, translating, summarizing, or explaining
        - Keywords: describe, explain, paraphrase, summarize
        3. Application: Using learned information in new concrete situations to solve problems
        - Keywords: apply, demonstrate, solve, use, implement
        4. Analysis: Breaking down information into parts, examining relationships, distinguishing facts from inferences
        - Keywords: analyze, compare, contrast, differentiate, examine
        5. Synthesis: Combining elements to form a new whole, proposing solutions, or designing new approaches
        - Keywords: create, design, propose, formulate, integrate
        6. Evaluation: Making judgments based on criteria and standards through checking and critiquing
        - Keywords: assess, critique, defend, evaluate, justify

        **Classification Rules:**
        - Output ONLY the lowercase label name (knowledge, comprehension, application, analysis, synthesis, evaluation)
        - Choose the HIGHEST level that substantially applies
        - Ignore verb tense/stemming (e.g., "analyzed" = analysis)
        - If multiple levels apply, select the most complex

        **Query to Classify:**
        "{query}"
    """

    message = flan_classifier(messages)
    
    pred_labels.append(message[0]['generated_text'].lower())

100%|██████████| 600/600 [3:05:12<00:00, 18.52s/it]  


In [19]:
print(classification_report(label , [label_mapper[key.lower()] for key in pred_labels]))

              precision    recall  f1-score   support

           0       1.00      0.76      0.86       100
           1       0.98      0.40      0.57       100
           2       0.41      0.68      0.51       100
           3       0.88      0.65      0.75       100
           4       0.44      0.84      0.58       100
           5       0.96      0.53      0.68       100

    accuracy                           0.64       600
   macro avg       0.78      0.64      0.66       600
weighted avg       0.78      0.64      0.66       600

